# Notebook 04 — Scale-Free Networks and the Barabási-Albert Model

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 04 of 12  
**Time estimate:** 75 minutes

> Most proteins in a PPI network have 1–3 partners. A handful have hundreds.
> This isn't random — it's the signature of preferential attachment, the mechanism
> that generates hubs and scale-free topology.

---
## Step 1 — Motivation

Erdős-Rényi graphs have a Poisson degree distribution — most nodes have roughly the
same degree. Real biological networks don't: they have heavy-tailed (power-law)
degree distributions with a few "hub" nodes that are vastly more connected than
average. Barabási and Albert (1999) showed this arises from *growth* + *preferential
attachment* — a mechanism that also explains why the internet, citation networks, and
metabolic networks all share this property.

---
## Step 2 — Intuition

**Preferential attachment:** "The rich get richer."
When a new protein evolves (or is duplicated), it is more likely to interact with
already highly-connected proteins — because they have more binding surfaces, are more
centrally located, or are more likely to be encountered in the cell.

Result: a **power-law degree distribution** $P(k) \propto k^{-\gamma}$, typically
$\gamma \approx 2$–$3$ for biological networks.

A network with a power-law degree distribution is called **scale-free** because the
distribution looks the same on any scale — there is no characteristic degree (unlike
Poisson, which peaks at $\langle k \rangle$).

**Hub proteins:** The few nodes with very high degree. In PPI networks, hub proteins
are often:
- Essential (knockouts are lethal) — "hub lethality" correlation
- Involved in many pathways (pleiotropic)
- Disease genes (targets of viruses, drugs)

---
## Step 3 — Biological Background

**Evidence for scale-free topology in biology:**
- E. coli PPI network: $\gamma \approx 2.4$
- Yeast two-hybrid PPI: $\gamma \approx 2.5$
- Metabolic networks (substrate graph): $\gamma \approx 2.2$

**Hub lethality:** In yeast PPI networks, ~93% of proteins encoded by essential genes
have degree ≥ 5; most essential proteins are hubs. This suggests evolutionary pressure
to maintain high-degree proteins.

**Robustness and fragility (Barabási and Albert, 2000):**
- Random node removal: scale-free networks are highly robust — removing a random node
  almost certainly removes a low-degree node, so the network stays connected
- Targeted removal of hubs: catastrophic — removing even a few top hubs
  fragments the network
- This duality is called **robust yet fragile** and is a signature of scale-free topology

**Network evolution mechanisms:**
1. Gene duplication → both copies start with same interactions → diverge
2. Domain recruitment → a domain present in a hub gets incorporated into new proteins
3. Both produce preferential-attachment-like dynamics

---
## Step 4 — Mathematical Explanation

**Barabási-Albert (BA) model:**
1. Start with a small seed graph (e.g., $m_0 = m$ fully connected nodes)
2. At each step, add a new node with $m$ edges
3. Each new edge attaches to existing node $i$ with probability:
$$\Pi(k_i) = \frac{k_i}{\sum_j k_j}$$

This is **preferential attachment**: probability proportional to current degree.

**Result (mean-field theory):**
$$P(k) \sim k^{-3}$$
The BA model generates scale-free networks with exponent $\gamma = 3$.

**Log-log degree distribution:** If $P(k) = C k^{-\gamma}$, then
$$\log P(k) = \log C - \gamma \log k$$
This is a straight line on a log-log plot — the empirical signature of scale-free topology.

**Power-law fit (maximum likelihood):**
$$\hat{\gamma} = 1 + n \left( \sum_{i=1}^n \ln \frac{k_i}{k_{\min} - 0.5} \right)^{-1}$$
where the sum is over all nodes with $k_i \geq k_{\min}$.

In [ ]:
# Step 6 — Python: BA model from scratch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def barabasi_albert(n_nodes, m, seed=42):
    """
    Grow a Barabási-Albert scale-free graph.
    
    Parameters
    ----------
    n_nodes : int
        Target total number of nodes (including seed)
    m : int
        Number of edges added per new node
    
    Returns
    -------
    adj : dict[int, set]
    edges : list of (u, v)
    """
    rng = np.random.default_rng(seed)
    # Seed: fully connected graph with m+1 nodes
    seed_n = m + 1
    adj = {i: set() for i in range(seed_n)}
    edges = []
    for i in range(seed_n):
        for j in range(i+1, seed_n):
            adj[i].add(j)
            adj[j].add(i)
            edges.append((i, j))
    
    # Grow network: add one node at a time
    # Maintain a flat list of all endpoints (stub list) for O(1) preferential attachment
    # Each edge (u,v) contributes u and v once → sampling stub ∝ degree
    stub_list = []
    for u, v in edges:
        stub_list += [u, v]
    
    for new_node in range(seed_n, n_nodes):
        adj[new_node] = set()
        targets = set()
        while len(targets) < m:
            candidate = rng.choice(stub_list)
            if candidate != new_node:
                targets.add(candidate)
        for t in targets:
            adj[new_node].add(t)
            adj[t].add(new_node)
            edges.append((new_node, t))
            stub_list += [new_node, t]  # add both endpoints
    
    return adj, edges


# Build BA network: N=500, m=2 → each new node adds 2 edges
adj_ba, edges_ba = barabasi_albert(500, m=2)
degrees_ba = np.array([len(adj_ba[v]) for v in adj_ba])
print(f'BA network: N={len(adj_ba)}, E={len(edges_ba)}')
print(f'Mean degree: {degrees_ba.mean():.2f}')
print(f'Max degree (largest hub): {degrees_ba.max()}')
print(f'Top 5 hubs (degree): {sorted(degrees_ba, reverse=True)[:5]}')

# Power-law exponent (MLE estimate)
def powerlaw_exponent_mle(degrees, k_min=2):
    """Maximum likelihood exponent for P(k) ~ k^(-gamma), k >= k_min."""
    k = degrees[degrees >= k_min].astype(float)
    n = len(k)
    gamma_hat = 1 + n / np.sum(np.log(k / (k_min - 0.5)))
    return gamma_hat

gamma_mle = powerlaw_exponent_mle(degrees_ba)
print(f'\nMLE power-law exponent γ = {gamma_mle:.3f}')
print(f'(BA model theoretical: γ = 3.0)')

In [ ]:
# Compare ER vs BA degree distributions
from collections import defaultdict

def erdos_renyi(n, p, seed=42):
    rng = np.random.default_rng(seed)
    adj = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(i+1, n):
            if rng.random() < p:
                adj[i].add(j)
                adj[j].add(i)
    return adj

# ER with same mean degree as BA
mean_k_ba = degrees_ba.mean()
p_er = mean_k_ba / (len(adj_ba) - 1)
adj_er = erdos_renyi(len(adj_ba), p_er)
degrees_er = np.array([len(adj_er[v]) for v in adj_er])

# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Degree distribution comparison (linear scale)
ax = axes[0]
bins = np.arange(0, degrees_ba.max()+2)
counts_ba = np.bincount(degrees_ba, minlength=bins.max())
counts_er = np.bincount(degrees_er, minlength=bins.max())
ax.plot(bins[1:len(counts_ba)], counts_ba[1:]/len(adj_ba), 'tomato', marker='o',
          ms=3, lw=1, label='BA (scale-free)')
ax.plot(bins[1:len(counts_er)], counts_er[1:]/len(adj_er), 'steelblue', marker='s',
          ms=3, lw=1, label='ER (random)')
ax.set_xlabel('Degree k')
ax.set_ylabel('P(k)')
ax.set_title('A. Degree distribution (linear)')
ax.legend()
ax.set_xlim(0, 30)

# Panel B: Log-log degree distribution
ax = axes[1]
k_vals_ba = np.unique(degrees_ba)
p_vals_ba = np.array([np.sum(degrees_ba == k) / len(degrees_ba) for k in k_vals_ba])
k_vals_er = np.unique(degrees_er)
p_vals_er = np.array([np.sum(degrees_er == k) / len(degrees_er) for k in k_vals_er])
ax.scatter(k_vals_ba, p_vals_ba, color='tomato', s=20, label='BA', zorder=3)
ax.scatter(k_vals_er, p_vals_er, color='steelblue', s=20, label='ER', zorder=3, marker='s')
# Power-law fit line
k_fit = np.linspace(2, degrees_ba.max(), 100)
c_fit = p_vals_ba[np.argmin(np.abs(k_vals_ba - 2))]
ax.plot(k_fit, c_fit * (k_fit/2)**(-gamma_mle), 'k--', lw=1,
          label=f'Power-law γ={gamma_mle:.2f}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Degree k (log)')
ax.set_ylabel('P(k) (log)')
ax.set_title('B. Log-log degree distribution\n(straight line = scale-free)')
ax.legend(fontsize=8)

# Panel C: Attack tolerance simulation
def largest_component_size(adj):
    visited = set()
    max_size = 0
    for start in adj:
        if start in visited:
            continue
        # BFS
        queue = [start]
        comp = set()
        while queue:
            u = queue.pop()
            if u in comp:
                continue
            comp.add(u)
            for v in adj.get(u, set()):
                if v not in comp:
                    queue.append(v)
        visited |= comp
        max_size = max(max_size, len(comp))
    return max_size

n_remove = 30
nodes_by_degree = sorted(adj_ba.keys(), key=lambda v: len(adj_ba[v]), reverse=True)
rng = np.random.default_rng(0)
nodes_random = rng.permutation(list(adj_ba.keys()))[:n_remove]

lcc_targeted, lcc_random = [], []
adj_t = {v: set(adj_ba[v]) for v in adj_ba}
adj_r = {v: set(adj_ba[v]) for v in adj_ba}

for i in range(n_remove):
    # Targeted: remove highest-degree node
    hub = nodes_by_degree[i]
    for nbr in list(adj_t.get(hub, set())):
        adj_t[nbr].discard(hub)
    adj_t.pop(hub, None)
    lcc_targeted.append(largest_component_size(adj_t) / len(adj_ba))
    
    # Random: remove random node
    rand_node = nodes_random[i]
    for nbr in list(adj_r.get(rand_node, set())):
        adj_r[nbr].discard(rand_node)
    adj_r.pop(rand_node, None)
    lcc_random.append(largest_component_size(adj_r) / len(adj_ba))

ax = axes[2]
frac_removed = np.arange(1, n_remove+1) / len(adj_ba)
ax.plot(frac_removed, lcc_targeted, 'tomato', lw=2, marker='o', ms=4, label='Targeted (hub removal)')
ax.plot(frac_removed, lcc_random, 'steelblue', lw=2, marker='s', ms=4, label='Random removal')
ax.set_xlabel('Fraction of nodes removed')
ax.set_ylabel('Largest component / N')
ax.set_title('C. Attack tolerance\n(robust to random, fragile to targeted)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)

plt.suptitle('Module 12 NB04: Scale-Free Networks and Barabási-Albert Model',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('scale_free_ba.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the BA model with $m=1$, $m=3$, $m=5$. How does $m$ affect the
   power-law exponent?
2. Identify the top-10 hub nodes in the BA network. What fraction of all edges
   are incident to these 10 nodes?
3. Modify the preferential attachment to be **uniform** (random, not by degree).
   What happens to the degree distribution?
4. What is the connection between "hub lethality" in yeast PPI networks and the
   attack tolerance simulated in Panel C?

---
## Step 10 — Quiz

1. What is preferential attachment? State it mathematically.
2. What does a power-law degree distribution look like on a log-log plot?
3. What is the predicted exponent $\gamma$ for the BA model?
4. Why are scale-free networks robust to random failure but fragile to targeted attack?
5. What biological mechanism is analogous to preferential attachment in protein evolution?

---
## Step 12 — Reflection

> *[In one paragraph: explain why the existence of hub proteins in a PPI network is
> both an advantage (robustness to random mutations) and a vulnerability (drug targets,
> viral exploitation), citing the attack tolerance simulation.]*

---
*Next: `05_network_centrality_measures.ipynb`*